# BOA Evaluation Task

### GRU Backbone Implemetnation

by Sardaruddin Syed (ssardaruddin2002@gmail.com)

In [15]:
import sys
import re
import pathlib

import pandas as pd
import numpy as np

import torch
import torch.nn as nn

In [2]:
!git clone https://github.com/iamakamen/boa-constrictor.git boa

Cloning into 'boa'...
remote: Enumerating objects: 358, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 358 (delta 68), reused 61 (delta 61), pack-reused 259 (from 2)
Receiving objects: 100% (358/358), 92.41 MiB | 42.74 MiB/s, done.
Resolving deltas: 100% (126/126), done.


In [8]:
%cd boa

/content/boa


In [9]:
!git checkout feat/gru-backbone

Branch 'feat/gru-backbone' set up to track remote branch 'feat/gru-backbone' from 'origin'.
Switched to a new branch 'feat/gru-backbone'


In [10]:
%%capture

!python -m pip install -qq --upgrade pip setuptools wheel

!apt-get update -qq
!apt-get install -y -qq build-essential cmake ninja-build pkg-config

!python -m pip install -qq pybind11 cmake ninja setuptools wheel
!python -m pip install -qq "torch>=2.0.0" torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [11]:
%%capture

!grep -v -E "^(torch|mamba-ssm|causal-conv1d)" requirements.txt > /content/boa/reqs_partial.txt
!python -m pip install -qq -r /content/boa/reqs_partial.txt

In [12]:
print("Python version:", sys.version)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version reported by torch:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch version: 2.10.0+cu128
CUDA available: True
CUDA version reported by torch: 12.8
GPU: Tesla T4


In [26]:
!mkdir -p logs

In [29]:
!python3 main.py --config experiments/cms_experiment/cms_experiment.yaml --compress-only --show-timings | tee logs/compression_log.txt

/content/boa/boa.py:231: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  t = torch.from_numpy(np.ascontiguousarray(sl)).unsqueeze(0)
cuda
Read 49920000 bytes from /content/boa/experiments/cms_experiment/CMS_DATA_float32.bin in 0.03s
Loading pre-trained model from /content/boa/experiments/cms_experiment/cms_experiment_final_model_fp32.pt
Skipping training (final model or explicit path provided).
Model loaded in 0.34s
Starting compression...
[compress] gpu_streams=5000 (chunk_len=4992, free=14.4GiB)
256
Compress (GPU streams x5000): 100% 24370.1171875/24370.1171875 [00:09<00:00, 2608.23KB/s]
256
Compress (GPU streams x5

In [30]:
!python3 main.py --config experiments/cms_experiment/cms_experiment.yaml --decompress-only --show-timings | tee logs/decompression_log.txt

cuda
Read 49920000 bytes from /content/boa/experiments/cms_experiment/CMS_DATA_float32.bin in 0.03s
Loading pre-trained model from /content/boa/experiments/cms_experiment/cms_experiment_final_model_fp32.pt
Skipping training (final model or explicit path provided).
Model loaded in 0.34s
Starting decompression...
Total compressed size from disk: 50105268 bytes
[decompress] gpu_streams=5000 (free=14.4GiB)
Decompress (GPU streams x5000): 100% 24370.1171875/24370.1171875 [00:11<00:00, 2174.19KB/s]
Decompress (GPU streams x5000): 100% 24370.1171875/24370.1171875 [00:11<00:00, 2169.06KB/s]
Decompression complete in 23.10s

Timings:
  read_bytes: 0.03s
  load_model: 0.34s
  decompression: 23.10s


In [33]:
pd.set_option('display.float_format', '{:.2f}'.format)

exp = pathlib.Path("experiments/cms_experiment")
comp_log = pathlib.Path("logs/compression_log.txt")
decomp_log = pathlib.Path("logs/decompression_log.txt")

def get_size(name):
    p = exp / name
    return p.stat().st_size if p.exists() else None

original_size = get_size("CMS_DATA_float32.bin")
boa_size = get_size("cms_experiment.boa")
lzma_size = get_size("cms_experiment.lzma")
zlib_size = get_size("cms_experiment.zlib")

comp_text = comp_log.read_text()
decomp_text = decomp_log.read_text()

comp_match = re.search(r"compression:\s*([0-9.]+)s", comp_text, re.IGNORECASE)
decomp_match = re.search(r"decompression:\s*([0-9.]+)s", decomp_text, re.IGNORECASE)

compression_time_seconds = float(comp_match.group(1)) if comp_match else None
decompression_time_seconds = float(decomp_match.group(1)) if decomp_match else None

def ratio(orig, comp):
    return round(orig / comp, 2) if orig and comp else None

boa_ratio = ratio(original_size, boa_size)
lzma_ratio = ratio(original_size, lzma_size)
zlib_ratio = ratio(original_size, zlib_size)

def throughput(bytes_in, seconds):
    return round((bytes_in / 1e6) / seconds, 2)

compression_MBps = throughput(original_size, compression_time_seconds)
decompression_MBps = throughput(original_size, decompression_time_seconds)

In [34]:
df = pd.DataFrame({
    "metric": [
        "original_size_bytes",
        "boa_size_bytes",
        "lzma_size_bytes",
        "zlib_size_bytes",
        "boa_ratio",
        "lzma_ratio",
        "zlib_ratio",
        "compression_MBps",
        "decompression_MBps"
    ],
    "value": [
        original_size,
        boa_size,
        lzma_size,
        zlib_size,
        boa_ratio,
        lzma_ratio,
        zlib_ratio,
        compression_MBps,
        decompression_MBps
    ]
})

df

,metric,value
0,original_size_bytes,49920000.00
1,boa_size_bytes,50155324.00
2,lzma_size_bytes,15489644.00
3,zlib_size_bytes,19463909.00
4,boa_ratio,1.00
5,lzma_ratio,3.22
6,zlib_ratio,2.56
7,compression_MBps,2.50
8,decompression_MBps,2.16
